# H&M baseline: bảng feature theo khách hàng
Chạy `00_setup_colab.ipynb` để cài package, sau đó chạy notebook này. Kết quả chỉ có `customer_id` và 384 feature: 128 transaction, 128 text, 128 image. Không có nhãn churn hoặc dự đoán 0/1.

## 1. Mount Drive
Dùng cùng tài khoản đã mở thư mục Drive được chia sẻ trong `00_setup_colab.ipynb`. Chạy cell tìm dữ liệu ở notebook setup trước: cell đó tự tạo shortcut Project Master Thesis trong My Drive khi cần. Bảng feature được lưu vào `Project Master Thesis/Data/features`, nên tài khoản cần quyền ghi trong thư mục dùng chung. Không tải hoặc giải nén ZIP vào repository.

In [13]:
from pathlib import Path
from google.colab import drive

try:
    drive.mount('/content/drive', timeout_ms=300000)
except ValueError as exc:
    raise RuntimeError(
        'Google Drive mount failed. Reconnect the Colab runtime and authorize the account '
        'that can open the H&M folder.'
    ) from exc
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Drive mount completed, but /content/drive/MyDrive is unavailable.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Chọn dữ liệu và mốc quan sát
`OBSERVATION_END` dưới đây chỉ dùng để chạy thử pipeline; luận văn chưa chốt observation/prediction windows. `MAX_CUSTOMERS=16` là mẫu kỹ thuật, không dùng để báo cáo kết quả nghiên cứu.

In [14]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/content/multimodal-churn-fusion')
SRC_DIR = PROJECT_DIR / 'src'
if not (SRC_DIR / 'multimodal_churn').is_dir():
    raise FileNotFoundError('Run 00_setup_colab.ipynb first to clone the repository.')
os.chdir(PROJECT_DIR)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from multimodal_churn.utils.config import find_drive_dataset_paths, load_config

MY_DRIVE = Path('/content/drive/MyDrive')
DATASETS_ROOT_OVERRIDE = None  # Parent of 02_hm_fashion, if shortcut is elsewhere.
PROJECT_DATA_ROOT = MY_DRIVE / 'Project Master Thesis' / 'Data'
OBSERVATION_END = '2020-08-31'  # Demo only; not a finalized churn cutoff.
OBSERVATION_START = None
MAX_CUSTOMERS = 16

baseline_path = Path('configs/baseline.yaml')
config = load_config(baseline_path, dataset_path=Path('configs/datasets/hm.yaml'))
DATASETS_ROOT, paths = find_drive_dataset_paths(config, MY_DRIVE, DATASETS_ROOT_OVERRIDE)
archive_path = paths['archive']
if not PROJECT_DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f'{PROJECT_DATA_ROOT} is unavailable. Add the Project Master Thesis shortcut to My Drive '
        'and check access to its Data folder.'
    )
output_dir = PROJECT_DATA_ROOT / 'features' / f'hm_demo_{OBSERVATION_END}'
print('Dataset root:', DATASETS_ROOT)
print('Archive:', archive_path)
print('Output:', output_dir)

Dataset root: /content/drive/MyDrive/Project Master Thesis/Data/Datasets
Archive: /content/drive/MyDrive/Project Master Thesis/Data/Datasets/02_hm_fashion/h-and-m-personalized-fashion-recommendations.zip
Output: /content/drive/MyDrive/Project Master Thesis/Data/features/hm_demo_2020-08-31


## 3. Tạo bảng feature
Lần chạy đầu sẽ tải trọng số MiniLM và CLIP công khai. Transformer giao dịch và phép chiếu 128D chưa được huấn luyện; bảng này kiểm tra luồng vector và phép concat, chưa chứng minh khả năng dự đoán.

In [15]:
from multimodal_churn.feature_pipeline import build_hm_feature_table

table = build_hm_feature_table(
    config=config,
    archive_path=archive_path,
    observation_start=OBSERVATION_START,
    observation_end=OBSERVATION_END,
    max_customers=MAX_CUSTOMERS,
    output_dir=output_dir,
)
print('Rows, columns:', table.shape)
print('CSV:', output_dir / 'features.csv')

H&M transaction chunks: 0chunk [00:00, ?chunk/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

CLIP image batches:   0%|          | 0/42 [00:00<?, ?batch/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Rows, columns: (16, 385)
CSV: /content/drive/MyDrive/Project Master Thesis/Data/features/hm_demo_2020-08-31/features.csv


## 4. Kiểm tra bảng
Một dòng là một khách hàng. Thứ tự cột là transaction → text → image, mỗi khối 128 chiều. File `manifest.json` ghi mốc quan sát, độ phủ modality và trạng thái chưa huấn luyện.

In [16]:
import json
assert table.shape[1] == 1 + 3 * config['model']['common_embedding_dim']
assert 'label' not in table.columns and 'churn' not in table.columns
expected_files = ('features.csv', 'manifest.json', 'projection_matrices.npz', 'transaction_encoder.pt')
assert all((output_dir / name).is_file() for name in expected_files)
print('Saved files:')
for name in expected_files:
    print('-', output_dir / name)
print(json.loads((output_dir / 'manifest.json').read_text(encoding='utf-8')))
display(table[['customer_id', 'transaction_000', 'text_000', 'image_000']].head())

Saved files:
- /content/drive/MyDrive/Project Master Thesis/Data/features/hm_demo_2020-08-31/features.csv
- /content/drive/MyDrive/Project Master Thesis/Data/features/hm_demo_2020-08-31/manifest.json
- /content/drive/MyDrive/Project Master Thesis/Data/features/hm_demo_2020-08-31/projection_matrices.npz
- /content/drive/MyDrive/Project Master Thesis/Data/features/hm_demo_2020-08-31/transaction_encoder.pt
{'dataset': 'hm', 'archive_path': '/content/drive/MyDrive/Project Master Thesis/Data/Datasets/02_hm_fashion/h-and-m-personalized-fashion-recommendations.zip', 'observation_start': None, 'observation_end': '2020-08-31', 'sample_customers': 16, 'selection_note': 'First encountered customers; smoke sample, not a research cohort.', 'max_sequence_length': 32, 'customer_rows': 16, 'observed_transaction_rows': 392, 'text_articles_encoded': 334, 'image_articles_encoded': 331, 'text_customers_available': 16, 'image_customers_available': 16, 'feature_order': ['transaction', 'text', 'image'], 'fea

,customer_id,transaction_000,text_000,image_000
0,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.320572,0.018031,-0.049233
1,00007d2de826758b65a93dd24ce629ed66842531df6699...,0.247530,0.045993,-0.045451
2,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,0.181822,-0.018837,-0.031461
3,0008968c0d451dbc5a9968da03196fe20051965edde741...,0.254584,0.019683,-0.051396
4,000aa7f0dc06cd7174389e76c9e132a67860c5f65f9706...,0.233376,0.000945,-0.050194
